# When AI reads text but systems need exact identity

Five compact, local demonstrations of a single risky pattern: an AI makes a semantic judgment, but the system it controls needs exact validation. All names and `.example` domains are fictional. All actions below are simulated with `print()`—no email, payment, file, or deployment is touched.

## Setup

If needed, run `%pip install transformers torch`. The first run downloads the public `Qwen/Qwen2.5-0.5B-Instruct` model.

In [1]:
import re
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

def ask(prompt):
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            **inputs, do_sample=False, max_new_tokens=16,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

def verdict(raw, choices):
    upper = raw.upper().replace('_', ' ')
    for choice in sorted(choices, key=len, reverse=True):
        if re.search(rf'\b{re.escape(choice)}\b', upper):
            return choice
    return 'UNPARSEABLE'

def show(title, value, ai, system):
    print(f'\n{title}')
    print(f'  Input:  {value!r}')
    print(f'  AI:     {ai}')
    print(f'  System: {system}')
    result = '⛔ HOLD — disagreement' if ai != system else '✓ Agreement'
    print(f'  Result: {result}')

print(f'Loaded {MODEL}')

/private/tmp/agentic-text-lab/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/private/tmp/agentic-text-lab/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Qwen/Qwen2.5-0.5B-Instruct


## 1. Invoice processing

An invoice workflow accepts references with exactly seven digits. This should be code, not an LLM decision. A false rejection can delay payment; a false acceptance can route a malformed invoice into automation.

In [2]:
reference = 'INV-4837291'
raw = ask(f'''
An invoice reference is VALID only when it contains exactly 7 decimal digits.
Return exactly VALID or INVALID. Reference: {reference}
''')
ai = verdict(raw, ['VALID', 'INVALID'])
system = 'VALID' if sum(c.isdecimal() for c in reference) == 7 else 'INVALID'
show('1. Invoice reference', reference, ai, system)


1. Invoice reference
  Input:  'INV-4837291'
  AI:     INVALID
  System: VALID
  Result: ⛔ HOLD — disagreement


## 2. Phishing triage

An email agent is asked whether a payment request is from `northwind.example`. The lookalike has a Cyrillic `о`. An inbox should use authenticated sender and canonical-domain checks—not a model's reading of a rendered name.

In [3]:
trusted = 'northwind.example'
sender = 'n\u043erthwind.example'  # Cyrillic o
raw = ask(f'''
Is {sender} associated with trusted supplier {trusted}?
Treat visually similar names as the same brand.
Return exactly TRUSTED or UNTRUSTED.
''')
ai = verdict(raw, ['TRUSTED', 'UNTRUSTED'])
system = 'TRUSTED' if sender == trusted else 'UNTRUSTED'
show('2. Email sender', sender, ai, system)


2. Email sender
  Input:  'nоrthwind.example'
  AI:     TRUSTED
  System: UNTRUSTED
  Result: ⛔ HOLD — disagreement


## 3. Vendor payment routing

An invoice is allowed into a payment workflow only for an approved vendor. The submitted name has a zero-width space. The model may read the same vendor; the ERP must not.

In [4]:
approved = 'Acme Supplies'
vendor = 'Acme\u200b Supplies'  # zero-width space
raw = ask(f'''
Is {vendor} the approved vendor {approved}?
Treat visually equivalent names as the same vendor.
Return exactly APPROVED or UNKNOWN.
''')
ai = verdict(raw, ['APPROVED', 'UNKNOWN'])
system = 'APPROVED' if vendor == approved else 'UNKNOWN'
show('3. Payment vendor', vendor, ai, system)


3. Payment vendor
  Input:  'Acme\u200b Supplies'
  AI:     APPROVED
  System: UNKNOWN
  Result: ⛔ HOLD — disagreement


## 4. Access-request routing

A helpdesk agent routes requests for the privileged `Finance-Approvers` group. The request contains a zero-width space. If an AI classifier decides that this is the privileged group, it must still not authorize the raw group name.

In [5]:
privileged = 'Finance-Approvers'
requested = 'Finance\u200b-Approvers'  # zero-width space
raw = ask(f'''
A request may enter the privileged workflow only for {privileged}.
Treat visually similar names as the same group.
Return exactly APPROVED or HOLD. Requested group: {requested}
''')
ai = verdict(raw, ['APPROVED', 'HOLD'])
system = 'APPROVED' if requested == privileged else 'HOLD'
show('4. Access group', requested, ai, system)


4. Access group
  Input:  'Finance\u200b-Approvers'
  AI:     APPROVED
  System: HOLD
  Result: ⛔ HOLD — disagreement


## 5. Deployment environment

A release agent treats `production` as a protected environment. The submitted name swaps in a Cyrillic `о`. The right behavior is to protect only the canonical environment ID, never a model's semantic match.

In [6]:
protected = 'production'
environment = 'pr\u043eduction'  # Cyrillic o
raw = ask(f'''
Is {environment} the protected environment {protected}?
Treat visually similar names as the same environment.
Return exactly PROTECTED or OTHER.
''')
ai = verdict(raw, ['PROTECTED', 'OTHER'])
system = 'PROTECTED' if environment == protected else 'OTHER'
show('5. Deployment target', environment, ai, system)


5. Deployment target
  Input:  'prоduction'
  AI:     PROTECTED
  System: OTHER
  Result: ⛔ HOLD — disagreement


## Takeaway

An LLM can be useful for summaries, explanations, and low-risk routing. It should not be the final authority for an invoice rule, sender identity, vendor ID, access group, or deployment target. Validate those values at the tool boundary with exact, documented rules.